# Human-in-the-Loop con Azure AI Assistants\nEste notebook demuestra cómo implementar un escenario de human-in-the-loop utilizando el SDK de Azure AI Assistants. El flujo se detendrá para solicitar la intervención del usuario antes de continuar.

In [ ]:
# Importar librerías necesarias
import asyncio
import os
from dotenv import load_dotenv
from azure.ai.assistant.aio import AssistantClient
from azure.ai.assistant.models import Assistant, AssistantThread, ThreadMessage

## Cargar Variables de Entorno y Configurar Cliente

In [ ]:
load_dotenv()

client = AssistantClient(
    endpoint=os.environ.get("AZURE_AI_PROJECT_ENDPOINT"),
)

## Crear el Asistente y el Hilo de Conversación

In [ ]:
async def create_assistant_and_thread():
    assistant = await client.create_assistant(
        model=os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME"),
        name="Asistente de Aprobación",
        instructions="Eres un asistente que ayuda a gestionar un proceso de aprobación. Primero, resume la solicitud y luego pide la confirmación del usuario."
    )
    thread = await client.create_thread()
    return assistant, thread

assistant, thread = asyncio.run(create_assistant_and_thread())

## Ejecutar el Flujo con Intervención Humana

In [ ]:
async def main():
    # Añadir mensaje inicial al hilo
    await client.create_message(thread.id, content="Por favor, procesa la solicitud para la orden de compra #12345.")

    # Iniciar la ejecución y esperar la respuesta del asistente
    run = await client.create_run(assistant.id, thread.id)
    while run.status in ['in_progress', 'queued']:
        await asyncio.sleep(1)
        run = await client.get_run(thread.id, run.id)
    
    messages = await client.get_messages(thread.id)
    print(f"Asistente: {messages.data[0].content[0].text.value}")

    # Simular la entrada del usuario
    user_response = input("¿Apruebas esta solicitud? (sí/no): ")

    if user_response.lower() == 'sí':
        # Añadir la aprobación del usuario y continuar
        await client.create_message(thread.id, content="El usuario ha aprobado la solicitud.")
        run = await client.create_run(assistant.id, thread.id, instructions="Procede con la tarea principal.")
        while run.status in ['in_progress', 'queued']:
            await asyncio.sleep(1)
            run = await client.get_run(thread.id, run.id)
        messages = await client.get_messages(thread.id)
        print(f"Asistente: {messages.data[0].content[0].text.value}")
    else:
        print("Proceso cancelado por el usuario.")

    # Limpiar
    await client.delete_assistant(assistant.id)
    await client.delete_thread(thread.id)

if __name__ == "__main__":
    asyncio.run(main())